# Annealing / basin-hopping

In [2]:
import numpy as np
import os
import polars as pl
from scipy.optimize import basinhopping

## Load admissions data

In [3]:
path_to_msoa_stats = os.path.join('..', 'data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [4]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662


Check sum of admissions for Welsh areas:

In [5]:
df_stats.filter(df_stats['country'] == 'W')['admissions'].sum()

0.0

Welsh data is always zero so remove it.

In [6]:
df_stats = df_stats.filter(df_stats['country'] != 'W')

Recalculate total numbers of patients:

In [7]:
df_stats = df_stats.with_columns((pl.col('good_health') + pl.col('fair health') + pl.col('bad health')).alias('total_health'))

In [8]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526


Pick out column names for the health and age proportions:

In [9]:
health_numbers = ['good_health', 'fair health', 'bad health']
props_health = ['prop_good_health', 'prop_fair health', 'prop_bad health']
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]

Calculate numbers of patients in each age band:

In [10]:
age_numbers = []

for col in props_age:
    new_col = col.replace('_proportion', '')
    age_numbers.append(new_col)
    df_stats = df_stats.with_columns((pl.col(col) * pl.col('total_health')).alias(new_col))

In [11]:
df_stats[['total_health'] + props_age + age_numbers].head()

total_health,age_less65_proportion,age_65_proportion,age_70_proportion,age_75_proportion,age_over80_proportion,age_less65,age_65,age_70,age_75,age_over80
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8524,0.7872,0.0559,0.0528,0.0422,0.062,6710.0928,476.4916,450.0672,359.7128,528.488
6634,0.7467,0.0578,0.0774,0.0492,0.0692,4953.6078,383.4452,513.4716,326.3928,459.0728
7100,0.7729,0.0609,0.0582,0.0421,0.0661,5487.59,432.39,413.22,298.91,469.31
10127,0.8091,0.0465,0.0438,0.0367,0.0638,8193.7557,470.9055,443.5626,371.6609,646.1026
8526,0.7643,0.0597,0.067,0.0425,0.0662,6516.4218,509.0022,571.242,362.355,564.4212


## Set up bins for IMD scores

Use quantiles so that each bin contains 10% of the MSOA (to match plot notebook).

The resulting dictionary has keys for which quantile it is and values for the left side (smaller, minimum value in bin) of the IMD bin.

In [12]:
dict_quantiles = {}

for q in np.arange(0.0, 1.01, 0.2):
    v = df_stats['IMD2019Score'].quantile(q)
    # Flip the ranking because lower rank (more deprived) should be for
    # higher IMD scores (more deprived).
    q_store = 1.0 - q
    dict_quantiles[round(q_store, 1)] = round(v, 5)

In [13]:
dict_quantiles

{1.0: 2.2122,
 0.8: 10.369,
 0.6: 15.2564,
 0.4: 21.81475,
 0.2: 31.84075,
 0.0: 87.02675}

Pick out just the values for the left edges of the bins:

In [14]:
imd_bin_left_edges = {}
for k in list(dict_quantiles.keys())[:-1]:
    imd_bin_left_edges[k] = dict_quantiles[k]

In [15]:
imd_bin_left_edges

{1.0: 2.2122, 0.8: 10.369, 0.6: 15.2564, 0.4: 21.81475, 0.2: 31.84075}

Place MSOA into these bins:

In [16]:
# Make columns for the results with a placeholder value:
# df_stats = df_stats.with_columns(pl.lit(0).alias('imd_bin_min'))
# df_stats = df_stats.with_columns(pl.lit(0).alias('imd_bin_max'))
df_stats = df_stats.with_columns(pl.lit(0.0).alias('depriv_quantile_min'))
df_stats = df_stats.with_columns(pl.lit(0.0).alias('depriv_quantile_max'))

for q, quantile in enumerate(list(dict_quantiles.keys())[:-1]):
    # Pick out the bin edges:
    qmin = list(dict_quantiles.keys())[q+1]
    q0 = dict_quantiles[quantile]
    q1 = dict_quantiles[qmin]
    # Find a mask for the demographic data that contains
    # only MSOA with IMD scores in this bin.
    mask = (df_stats['IMD2019Score'] >= q0) & (df_stats['IMD2019Score'] < q1)
    if q == len(dict_quantiles) - 2:
        # Also allow values at the right edge of the final bin.
        mask = mask | (df_stats['IMD2019Score'] == q1)

    # Update the bin min/max values for these rows:
    df_stats = df_stats.with_columns(
        pl.when((mask))
        .then(qmin)         # replace with bin min
        .otherwise(pl.col('depriv_quantile_min'))  # otherwise keep the existing value
        .name.keep()
    )
    df_stats = df_stats.with_columns(
        pl.when((mask))
        .then(quantile)         # replace with bin min
        .otherwise(pl.col('depriv_quantile_max'))  # otherwise keep the existing value
        .name.keep()
    )

Check that data was binned correctly:

In [17]:
df_stats[['IMD2019Score', 'depriv_quantile_min', 'depriv_quantile_max']]

IMD2019Score,depriv_quantile_min,depriv_quantile_max
f64,f64,f64
16.924833,0.4,0.6
6.4704,0.8,1.0
13.7334,0.6,0.8
26.199857,0.2,0.4
11.7948,0.6,0.8
…,…,…
3.25925,0.8,1.0
7.29475,0.8,1.0
12.117,0.6,0.8


## Admissions prediction setup

In [18]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [19]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (yhat - y)**2.0
    # Sum of differences:
    sum_sqres = sqres.sum()
    return sum_sqres

Goodness check option 2: This calculates the mean absolute difference between the predicted and real admissions numbers:

In [20]:
def find_mean_abs_diff(yhat, y):
    # Difference from actual:
    absres = np.abs((yhat - y))
    # Mean of differences:
    mean_absres = absres.mean()
    return mean_absres

The following function combines the previous ones to do the main calculations we need:

In [21]:
def main_admissions(coeffs, args):
    x_lists = args[0]
    admissions = args[1]
    yhat = predict_admissions(x_lists, coeffs)
    # Use only one of the following options for checking the fit:
    # sum_sqres = find_square_residuals(yhat, admissions)
    sum_sqres = find_mean_abs_diff(yhat, admissions)
    return sum_sqres

To check accuracy, the following function calculates R-squared:

In [22]:
def calculate_rsquared(y, yhat):
    """This gives the same results as the sklearn built-in."""
    y_mean = y.mean()
    ss_res = ((yhat - y)**2.0).sum()
    ss_tot = ((y - y_mean)**2.0).sum()
    if ss_tot != 0.0:
        rsq = 1.0 - ss_res / ss_tot
    else:
        rsq = np.NaN
    return rsq

## Parameter setup

Which values are allowed for each coefficient?

In [23]:
# From minimiser before:
best_coeffs_by_depriv = {
    '0.0': [0.0006, 0.003, 0.004, 0.005, 0.017],
    '0.2': [0.0005, 0.002, 0.003, 0.004, 0.017],
    '0.4': [0.0004, 0.001, 0.002, 0.004, 0.017],
    '0.6': [0.0004, 0.001, 0.002, 0.004, 0.015],
    '0.8': [0.0003, 0.001, 0.002, 0.003, 0.015],
}

In [24]:
start_ranges_by_coeff = [
    [0.0001, 0.0007],
    [0.0007, 0.009],
    [0.0006, 0.006],
    [0.0007, 0.009],
    [0.012, 0.019]
]

## Basin hop

In [40]:
bounds_dict = {
    # str(float(1e-4)): {'min': 7e-5, 'max': 4e-4},
    # str(float(4e-4)): {'min': 1e-4, 'max': 7e-4},
    # str(float(7e-4)): {'min': 4e-4, 'max': 1e-3},
    # str(float(1e-3)): {'min': 7e-4, 'max': 4e-3},
    # str(float(4e-3)): {'min': 1e-3, 'max': 7e-3},
    # str(float(7e-3)): {'min': 4e-3, 'max': 1e-2},
    # str(float(1e-2)): {'min': 9e-3, 'max': 2e-2},
    #
    str(float(4e-4)): {'min': 1e-4, 'max': 7e-4},
    str(float(5e-4)): {'min': 2e-4, 'max': 8e-4},
    str(float(1e-3)): {'min': 7e-4, 'max': 4e-3},
    str(float(2e-3)): {'min': 8e-4, 'max': 5e-3},
    str(float(4e-3)): {'min': 1e-3, 'max': 7e-3},
    str(float(6e-3)): {'min': 3e-3, 'max': 9e-3},
    str(float(1.5e-2)): {'min': 9e-3, 'max': 2e-2},
    str(float(1.7e-2)): {'min': 9e-3, 'max': 2e-2},
}

In [41]:
# Store the results in this list:
list_dict_admissions_lines = []

q = 0
for quantile_right_prop, quantile_left in imd_bin_left_edges.items():
    mask = (df_stats['depriv_quantile_max'] == quantile_right_prop)
    quantile_right = dict_quantiles[list(dict_quantiles.keys())[q + 1]]
    quantile_left_prop = list(dict_quantiles.keys())[q + 1]
    # Keep only those MSOA:
    df_stats_here = df_stats.filter(mask)

    # Set up data for the optimiser.
    # Initial guess for health coefficients:
    # age_coeffs = [4e-4, 8e-4, 1e-3, 4e-3, 1.5e-2]  #[0.01 for a in age_numbers]
    age_coeffs = [4e-4, 1e-3, 2e-3, 4e-3, 1.5e-2]  #[0.01 for a in age_numbers]
    #     0.00005,  # P(stroke | good health)
    #     0.005,    # P(stroke | fair health)
    #     0.005     # P(stroke | bad health)
    # ]
    # MSOA data in the same order as those coefficients:
    x_lists = [df_stats_here[a] for a in age_numbers]
    # x_lists = [df_stats_here['good_health'], df_stats_here['fair health'], df_stats_here['bad health']]
    # Data for the optimiser function that should not be changed:
    args = [
        x_lists,
        df_stats_here['admissions']
    ]
    
    bounds_here = [(bounds_dict[str(float(a))]['min'], bounds_dict[str(float(a))]['max']) for a in age_coeffs]

    # Run the optimiser:
    opt_results = basinhopping(
        main_admissions,
        x0=age_coeffs,
        minimizer_kwargs = dict(
            args=args,
            # bounds=[(0.0, 1.0)] * len(age_coeffs),  # force results to lie between 0 and 1
            bounds=bounds_here,
            method='Nelder-Mead'
            ),
        # method='SLSQP',
    )
    # Pick out the resulting health coefficients:
    coeffs = opt_results['x']
    # Use these coefficients to predict the admissions and so
    # calculate r-squared for accuracy of the fit.
    admissions_predicted = predict_admissions(x_lists, coeffs)
    r2 = calculate_rsquared(df_stats_here['admissions'], admissions_predicted)

    # Store the results in a dictionary:
    dict_admissions_lines = {}
    dict_admissions_lines['depriv_quantile_min'] = quantile_left_prop
    dict_admissions_lines['depriv_quantile_max'] = quantile_right_prop
    # dict_admissions_lines['imd_bin_min'] = quantile_left
    # dict_admissions_lines['imd_bin_max'] = quantile_right
    for i, a in enumerate(age_numbers):
        dict_admissions_lines[f'coeff_{a}'] = round(coeffs[i], 7)
        # dict_admissions_lines['coeff_fair_health'] = round(coeffs[1], 7)
        # dict_admissions_lines['coeff_bad_health'] = round(coeffs[2], 7)
    dict_admissions_lines['rsquared'] = round(r2, 7)
    # Store in big list of all results:
    list_dict_admissions_lines.append(dict_admissions_lines)
    # Iterate for next go round the loop:
    q += 1

# Convert the list of dicts into a dataframe:
df_admissions_lines = pl.from_dicts(list_dict_admissions_lines)

In [42]:
with pl.Config(float_precision=7):
    display(df_admissions_lines[::-1])

depriv_quantile_min,depriv_quantile_max,coeff_age_less65,coeff_age_65,coeff_age_70,coeff_age_75,coeff_age_over80,rsquared
f64,f64,f64,f64,f64,f64,f64,f64
0.0000000,0.2000000,0.0005876,0.0040000,0.0049952,0.0010000,0.0172834,0.4948696
0.2000000,0.4000000,0.0004859,0.0035299,0.0043898,0.0010000,0.0155725,0.5806393
0.4000000,0.6000000,0.0003868,0.0022822,0.0008000,0.0039672,0.0167734,0.6438938
0.6000000,0.8000000,0.0003405,0.0007046,0.0009241,0.0070000,0.0146618,0.6121342
0.8000000,1.0000000,0.0001955,0.0031486,0.0008000,0.0029926,0.0151847,0.6268629


In [34]:

display(df_best_set_combo_r2)

start_option,depriv_quantile_min,depriv_quantile_max,coeff_age_less65,coeff_age_65,coeff_age_70,coeff_age_75,coeff_age_over80,rsquared,rsquared_prop,rsquared_prop_minus
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
86173,0.0,0.2,0.0006,0.003,0.004,0.005,0.017,0.496213,0.992539,0.003539
73589,0.2,0.4,0.0005,0.002,0.003,0.004,0.017,0.582614,0.996339,0.001339
61013,0.4,0.6,0.0004,0.001,0.002,0.004,0.017,0.645111,0.999442,0.003442
61011,0.6,0.8,0.0004,0.001,0.002,0.004,0.015,0.615163,1.0,0.003
49483,0.8,1.0,0.0003,0.001,0.002,0.003,0.015,0.628031,0.99963,0.00463
